In [6]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.cm as cm
import plotly.graph_objects as go
from scipy.stats import ortho_group

## Partial Manifold Alignment: 
### Motivation: 
Want to find a method that can take two different omics modalities, take their high dimensional data (represented as a manifold) and map 
it to a joint embbedding space. Once in this joint embedding space we try to optimize embeddings to align to their common structures from different modalities (i.e portions of the structure of the manifold are the same), but also retain information on how the structures are different as well. 

In [7]:
import numpy as np
import plotly.graph_objects as go

# Parametric trefoil knot
t = np.linspace(0, 2 * np.pi, 500)
x = np.sin(t) + 2 * np.sin(2 * t)
y = np.cos(t) - 2 * np.cos(2 * t)
z = -np.sin(3 * t)

# Sinusoidal radius modulation
r1 = 0.2 + 0.05 * np.sin(5 * t)

# Create 3D tube coordinates
theta = np.linspace(0, 2 * np.pi, 20)
X1, Y1, Z1 = [], [], []

for i in range(len(t)):
    dx, dy, dz = np.gradient(x), np.gradient(y), np.gradient(z)
    tangent = np.array([dx[i], dy[i], dz[i]])
    tangent /= np.linalg.norm(tangent)
    normal = np.cross(tangent, [0, 0, 1])
    normal /= np.linalg.norm(normal)
    binormal = np.cross(tangent, normal)

    for th in theta:
        circle = r1[i] * (np.cos(th) * normal + np.sin(th) * binormal)
        X1.append(x[i] + circle[0])
        Y1.append(y[i] + circle[1])
        Z1.append(z[i] + circle[2])

fig1 = go.Figure(data=[go.Scatter3d(
    x=X1, y=Y1, z=Z1,
    mode='markers',
    marker=dict(size=2, color=t, colorscale='Blues', opacity=0.8)
)])
fig1.update_layout(title='Trefoil Knot - Modality X (Sinusoidal Radius)', scene=dict(aspectmode='data'))
fig1.show()


In [8]:
# Gaussian bump radius modulation
r2 = 0.2 + 0.1 * np.exp(-((t - np.pi) ** 2) / 0.2)

X2, Y2, Z2 = [], [], []

for i in range(len(t)):
    dx, dy, dz = np.gradient(x), np.gradient(y), np.gradient(z)
    tangent = np.array([dx[i], dy[i], dz[i]])
    tangent /= np.linalg.norm(tangent)
    normal = np.cross(tangent, [0, 0, 1])
    normal /= np.linalg.norm(normal)
    binormal = np.cross(tangent, normal)

    for th in theta:
        circle = r2[i] * (np.cos(th) * normal + np.sin(th) * binormal)
        X2.append(x[i] + circle[0])
        Y2.append(y[i] + circle[1])
        Z2.append(z[i] + circle[2])

fig2 = go.Figure(data=[go.Scatter3d(
    x=X2, y=Y2, z=Z2,
    mode='markers',
    marker=dict(size=2, color=t, colorscale='Reds', opacity=0.8)
)])
fig2.update_layout(title='Trefoil Knot - Modality Y (Gaussian Bumps)', scene=dict(aspectmode='data'))
fig2.show()


This is the ideal case where the two manifolds are parallel and aligned. and we wont our embeddings to learn the correct alignment. But I don't know if this the right way to do this??? Ugh. I want to find a good initial representation into the space that is common. Like what would proteomics data look like in DNA SPACE? 

In [9]:
from sklearn.neighbors import kneighbors_graph

# Manifold 1 coordinates: (X1, Y1, Z1)
coords1 = np.vstack([X1, Y1, Z1]).T

# Create KNN graph (e.g., k=10)
knn_graph1 = kneighbors_graph(coords1, n_neighbors=20, mode='connectivity', include_self=False)

# knn_graph1 is a sparse adjacency matrix
print(knn_graph1.shape)

(10000, 10000)
